# 03 Stratified Random Coreset Baseline

Select `K_PER_CLASS` random examples from each AG News class, then train the same classifier.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from text_distillation.data import load_ag_news, make_tiny_subset
from text_distillation.data.datasets import get_label_names
from text_distillation.distillation import select_stratified_random
from text_distillation.model import train_text_classifier
from text_distillation.saving import create_run_dir, save_distilled_dataset, save_experiment_config, save_metrics
from text_distillation.utils import get_git_commit_hash, set_seed

In [ ]:
EXPERIMENT_NAME = "stratified_random_agnews_bert_base_k100"
DATASET_NAME = "ag_news"
METHOD_NAME = "stratified_random"
MODEL_NAME = "bert-base-uncased"
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
K_PER_CLASS = 100
MAX_LENGTH = 128
NUM_TRAIN_EPOCHS = 5.0
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
# T4 16GB starting point. If CUDA OOM happens, lower TRAIN_BATCH_SIZE to 32.
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
GRADIENT_ACCUMULATION_STEPS = 1
MIXED_PRECISION = "auto"
DATALOADER_NUM_WORKERS = 2
SEED = 42

MAX_EVAL_SAMPLES = None

set_seed(SEED)

In [ ]:
dataset = load_ag_news()
train_dataset = dataset["train"]
test_dataset = dataset["test"]

distilled_dataset = select_stratified_random(
    dataset=train_dataset,
    label_column=LABEL_COLUMN,
    k_per_class=K_PER_CLASS,
    seed=SEED,
)

if MAX_EVAL_SAMPLES is not None:
    test_dataset = make_tiny_subset(test_dataset, total_size=MAX_EVAL_SAMPLES, seed=SEED)

label_names = get_label_names(dataset["train"], LABEL_COLUMN)
print(distilled_dataset)
print(test_dataset)

In [ ]:
run_dir = create_run_dir(PROJECT_ROOT / "artifacts" / "runs", EXPERIMENT_NAME)
config = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_name": DATASET_NAME,
    "method_name": METHOD_NAME,
    "model_name": MODEL_NAME,
    "k_per_class": K_PER_CLASS,
    "max_length": MAX_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "mixed_precision": MIXED_PRECISION,
    "dataloader_num_workers": DATALOADER_NUM_WORKERS,
    "seed": SEED,
    "train_size": len(distilled_dataset),
    "eval_size": len(test_dataset),
    "git_commit": get_git_commit_hash(PROJECT_ROOT),
}
save_experiment_config(config, run_dir)
save_distilled_dataset(distilled_dataset, run_dir)

In [ ]:
model, metrics = train_text_classifier(
    train_dataset=distilled_dataset,
    eval_dataset=test_dataset,
    model_name=MODEL_NAME,
    output_dir=run_dir,
    text_column=TEXT_COLUMN,
    label_column=LABEL_COLUMN,
    num_labels=len(label_names),
    label_names=label_names,
    max_length=MAX_LENGTH,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    mixed_precision=MIXED_PRECISION,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    seed=SEED,
)
save_metrics(metrics, run_dir)
metrics